# Credit Card Fraud Detection - End-to-End Notebook

This notebook combines all Python modules into a single Google Colab notebook.

## Workflow
1. Install libraries
2. Import libraries
3. Load dataset
4. EDA
5. Preprocessing
6. Handle Imbalance (SMOTE)
7. Train Random Forest
8. Evaluate Model
9. Save Artifacts
10. Load Model
11. Predict New Transaction
12. Optional DNN Training


In [ ]:
!pip install -q imbalanced-learn tensorflow joblib seaborn

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import compute_class_weight


## data_loader.py


In [ ]:
# src/data_loader.py

import pandas as pd


def load_data(file_path):
    """
    Load credit card fraud dataset

    Parameters
    ----------
    file_path : str
        Path to CSV file

    Returns
    -------
    pd.DataFrame
    """

    print("=" * 50)
    print("Loading Dataset")
    print("=" * 50)

    df = pd.read_csv(file_path)

    print(f"Dataset Shape : {df.shape}")
    print(f"Rows          : {df.shape[0]}")
    print(f"Columns       : {df.shape[1]}")

    return df   


'''
if __name__ == "__main__":
    # Build path relative to repository root to avoid escape-sequence issues
    project_root = Path(__file__).resolve().parents[1]
    data_file = project_root / "data" / "raw" / "creditcard.csv"
    load_data(data_file)
    '''

## eda.py


In [ ]:
# ============================================================
# Credit Card Fraud Detection Project
# File: eda.py
#
# Purpose:
# Perform Exploratory Data Analysis (EDA)
#
# Business Goal:
# Understand fraud distribution, transaction patterns,
# data quality and class imbalance before model training.
# ============================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


def basic_data_overview(df):
    """
    Display basic dataset information.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    None
    """

    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)

    print("\nFirst 5 Rows:")
    print(df.head())

    print("\nDataset Shape:")
    print(df.shape)

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nData Types & Null Values:")
    df.info()

    print("\nSummary Statistics:")
    print(df.describe())


def check_missing_values(df):
    """
    Check missing values in dataset.
    """

    print("\n" + "=" * 60)
    print("MISSING VALUE ANALYSIS")
    print("=" * 60)

    missing_values = df.isnull().sum()

    print(missing_values)

    print("\nTotal Missing Values:")
    print(missing_values.sum())


def check_duplicates(df):
    """
    Check duplicate rows.
    """

    print("\n" + "=" * 60)
    print("DUPLICATE RECORD ANALYSIS")
    print("=" * 60)

    duplicate_count = df.duplicated().sum()

    print(f"Duplicate Rows : {duplicate_count}")


def class_distribution_analysis(df):
    """
    Analyze fraud vs non-fraud distribution.
    """

    print("\n" + "=" * 60)
    print("CLASS DISTRIBUTION ANALYSIS")
    print("=" * 60)

    class_counts = df["Class"].value_counts()

    class_percentage = (
        df["Class"]
        .value_counts(normalize=True)
        * 100
    )

    print("\nClass Counts:")
    print(class_counts)

    print("\nClass Percentage:")
    print(class_percentage)

    plt.figure(figsize=(8, 5))

    sns.countplot(
        x="Class",
        data=df
    )

    plt.title("Fraud vs Non-Fraud Transactions")
    plt.xlabel("Class")
    plt.ylabel("Count")

    plt.show()

    return class_counts, class_percentage


def amount_analysis(df):
    """
    Analyze transaction amount behaviour.
    """

    print("\n" + "=" * 60)
    print("TRANSACTION AMOUNT ANALYSIS")
    print("=" * 60)

    print(df["Amount"].describe())

    print("\nAmount Skewness:")
    print(df["Amount"].skew())

    # ---------------------------------------------------
    # Original Amount Distribution
    # ---------------------------------------------------

    plt.figure(figsize=(10, 5))

    sns.histplot(
        df["Amount"],
        bins=100,
        kde=True
    )

    plt.title("Transaction Amount Distribution")

    plt.show()

    # ---------------------------------------------------
    # Log Transformed Amount
    # ---------------------------------------------------

    plt.figure(figsize=(10, 5))

    sns.histplot(
        np.log1p(df["Amount"]),
        bins=100,
        kde=True
    )

    plt.title("Log Transformed Amount Distribution")

    plt.show()

    # ---------------------------------------------------
    # Fraud vs Non-Fraud Amount Comparison
    # ---------------------------------------------------

    plt.figure(figsize=(10, 5))

    sns.boxplot(
        x="Class",
        y="Amount",
        data=df
    )

    plt.title(
        "Transaction Amount by Fraud Class"
    )

    plt.show()


def correlation_analysis(df):
    """
    Generate correlation heatmap.
    """

    print("\n" + "=" * 60)
    print("CORRELATION ANALYSIS")
    print("=" * 60)

    plt.figure(figsize=(16, 12))

    correlation_matrix = df.corr()

    sns.heatmap(
        correlation_matrix,
        cmap="coolwarm",
        center=0
    )

    plt.title("Feature Correlation Heatmap")

    plt.show()


def generate_eda_report(df):
    """
    Run complete EDA workflow.
    """

    print("\n")
    print("=" * 70)
    print("STARTING EXPLORATORY DATA ANALYSIS")
    print("=" * 70)

    basic_data_overview(df)

    check_missing_values(df)

    check_duplicates(df)

    class_distribution_analysis(df)

    amount_analysis(df)

    correlation_analysis(df)

    print("\n")
    print("=" * 70)
    print("EDA COMPLETED SUCCESSFULLY")
    print("=" * 70)

## preprocessing.py


In [ ]:
# src/preprocessing.py

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler


def preprocess_data(df):

    print("=" * 50)
    print("Preprocessing Data")
    print("=" * 50)

    X = df.drop("Class", axis=1)
    y = df["Class"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    amount_scaler = RobustScaler()

    X_train["Amount"] = amount_scaler.fit_transform(
        X_train[["Amount"]]
    )

    X_test["Amount"] = amount_scaler.transform(
        X_test[["Amount"]]
    )

    feature_scaler = StandardScaler()

    columns_to_scale = [
        col for col in X_train.columns
        if col != "Amount"
    ]

    X_train[columns_to_scale] = feature_scaler.fit_transform(
        X_train[columns_to_scale]
    )

    X_test[columns_to_scale] = feature_scaler.transform(
        X_test[columns_to_scale]
    )

    print(f"Training Shape : {X_train.shape}")
    print(f"Testing Shape  : {X_test.shape}")

    return (
        X_train,
        X_test,
        y_train,
        y_test,
        amount_scaler,
        feature_scaler
    )

## imbalance_handler.py


In [ ]:
# src/imbalance_handler.py

from imblearn.over_sampling import SMOTE


def apply_smote(X_train, y_train):

    print("=" * 50)
    print("Applying SMOTE")
    print("=" * 50)

    print("Before SMOTE")
    print(y_train.value_counts())

    smote = SMOTE(
        random_state=42
    )

    X_train_resampled, y_train_resampled = smote.fit_resample(
        X_train,
        y_train
    )

    print("After SMOTE")
    print(y_train_resampled.value_counts())

    return X_train_resampled, y_train_resampled

## train_rf.py


In [ ]:
# src/train_rf.py

from sklearn.ensemble import RandomForestClassifier


def train_random_forest(
    X_train,
    y_train
):

    print("=" * 50)
    print("Training Random Forest")
    print("=" * 50)

    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(
        X_train,
        y_train
    )

    print("Random Forest Training Completed")

    return rf_model

## train_dnn.py


In [ ]:
# src/train_dnn.py

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import compute_class_weight
import numpy as np


def build_dnn(input_dim):

    model = Sequential()

    model.add(
        Dense(
            128,
            activation="relu",
            input_shape=(input_dim,)
        )
    )

    model.add(BatchNormalization())

    model.add(Dropout(0.30))

    model.add(Dense(64, activation="relu"))

    model.add(Dropout(0.20))

    model.add(Dense(32, activation="relu"))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "Precision",
            "Recall"
        ]
    )

    return model


def train_dnn(
    X_train,
    y_train,
    X_val=None,
    y_val=None
):

    model = build_dnn(
        X_train.shape[1]
    )

    early_stopping = EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    # compute class weights to handle imbalance without resampling
    classes = np.unique(y_train)
    cw_vals = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight = {int(cls): float(val) for cls, val in zip(classes, cw_vals)}

    fit_kwargs = dict(
        x=X_train,
        y=y_train,
        epochs=50,
        batch_size=256,
        callbacks=[early_stopping],
        class_weight=class_weight,
        verbose=1
    )

    if X_val is not None and y_val is not None:
        fit_kwargs["validation_data"] = (X_val, y_val)
    else:
        # fall back to small validation split if explicit val not provided
        fit_kwargs["validation_split"] = 0.20

    history = model.fit(**fit_kwargs)

    return model, history

## evaluate.py


In [ ]:
# src/evaluate.py

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)


def evaluate_sklearn_model(
    model,
    X_test,
    y_test
):

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob >= 0.50).astype(int)

    print("=" * 50)
    print("Model Evaluation")
    print("=" * 50)

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

    print(
        "ROC AUC :",
        roc_auc_score(
            y_test,
            y_prob
        )
    )

    print(
        "Confusion Matrix"
    )

    print(
        confusion_matrix(
            y_test,
            y_pred
        )
    )

## save_artifacts.py


In [ ]:
# src/save_artifacts.py

import joblib


def save_random_forest(
    model,
    file_path
):

    joblib.dump(
        model,
        file_path
    )

    print(
        f"Model Saved : {file_path}"
    )


def save_scaler(
    scaler,
    file_path
):

    joblib.dump(
        scaler,
        file_path
    )

    print(
        f"Scaler Saved : {file_path}"
    )

## model_loader.py


In [ ]:
# src/model_loader.py

import joblib


def load_model(
    model_path
):

    return joblib.load(
        model_path
    )


def load_scaler(
    scaler_path
):

    return joblib.load(
        scaler_path
    )

## predict.py


In [ ]:
# src/predict.py

import numpy as np


def predict_transaction(
    model,
    transaction
):

    probability = model.predict_proba(
        transaction
    )[0][1]

    prediction = int(
        probability >= 0.50
    )

    return {
        "fraud_probability": round(
            probability,
            4
        ),
        "prediction": prediction
    }

# Run Complete Pipeline


In [ ]:

# ==========================================
# MAIN EXECUTION FLOW
# ==========================================

# Upload creditcard.csv into Colab and update path
DATA_PATH = "/content/creditcard.csv"

df = load_data(DATA_PATH)

generate_eda_report(df)

X_train, X_test, y_train, y_test, amount_scaler, feature_scaler = preprocess_data(df)

X_train_smote, y_train_smote = apply_smote(X_train, y_train)

rf_model = train_random_forest(X_train_smote, y_train_smote)

evaluate_sklearn_model(
    rf_model,
    X_test,
    y_test
)

save_random_forest(
    rf_model,
    "random_forest_model.pkl"
)

save_scaler(
    amount_scaler,
    "amount_scaler.pkl"
)

save_scaler(
    feature_scaler,
    "feature_scaler.pkl"
)


# Prediction Example


In [ ]:

loaded_model = load_model("random_forest_model.pkl")

sample_transaction = X_test.iloc[[0]]

result = predict_transaction(
    loaded_model,
    sample_transaction
)

print(result)


# Optional DNN Training


In [ ]:

dnn_model, history = train_dnn(
    X_train,
    y_train
)
